In [1]:
import datetime
import os
import time

import polars as pl

from alftrade import (
    resolve_symbols, pool_label, symbol_names,
    fetch_daily_to_parquet,
)

# ── 控制面板 ──
POOL_SPECS = [                      # §2 逐个解析的股票池规格
    "csi300:10",                    # 沪深300 前 10
    "market_cap:10",                # 总市值前 10
    "market_cap_small:10",          # 总市值后 10
    "industry:酒",                  # 证监会行业/名称关键词(tushare)
    "valuation:pe_low:10",          # 静态市盈率最低 10
    "valuation:pe_ttm:10",          # 市盈率(TTM)最低 10
    "valuation:pb_high:10",         # 市净率最高 10
    "valuation:dy:10",              # 股息率最高 10(默认方向 high)
    ["600519", "000858", "000568", "600809", "002304"],   # 自选列表
]
FETCH_BARS = False                  # True → §6 为最后一个池子拉取日线并缓存
print("stock-pool tools ready")

stock-pool tools ready


## 1. 股票池规格总览

`resolve_symbols(spec)` 是所有股票池的统一入口:一个字符串规格 → 6 位代码列表。
`pool_label(spec)` 是纯本地(无网络)的对应标签,供回测/日报标注用。

| 规格 | 含义 | 数据源 | 备注 |
|---|---|---|---|
| `"csi300:N"` | 沪深300 前 N 只(按代码升序) | 中证(akshare `index_stock_cons`) | 稳定,秒级 |
| `"market_cap:N"` | 总市值前 N | tushare `daily_basic` → 东财 → 新浪逐股兜底 | tushare 秒级(日缓存) |
| `"market_cap_small:N"` | 总市值后 N | 同上 | 同上 |
| `"industry:关键词"` | 证监会行业或名称含关键词 | tushare `stock_basic` | 日缓存;分类较粗(`白酒`→用`酒`) |
| `"valuation:pe_low:N"` | 静态 PE 最低 N(仅正 PE) | tushare `daily_basic`(乐咕兜底) | 每日缓存,秒级 |
| `"valuation:pe_ttm:N"` | 市盈率(TTM)最低 N | 同上 | 同上 |
| `"valuation:pb_low:N"` | 市净率最低 N | 同上 | 同上 |
| `"valuation:dy:N"` | 股息率最高 N(dy 默认 high) | 同上 | 同上 |
| `"600519,000001"` | 逗号分隔自选 | 无 | — |
| `["600519", ...]` | 列表自选 | 无 | — |

`valuation:` 的完整语法是 `valuation:<指标>[:<方向>]:<N>`,指标 ∈ {pe, pe_ttm, pb, dy},
方向 ∈ {low, high};省略方向时 pe/pe_ttm/pb 默认 `low`,dy 默认 `high`。

In [2]:
# ── §2 逐个解析:规格 → 代码 → 名称 ──
resolved = {}
for spec in POOL_SPECS:
    t0 = time.time()
    try:
        codes = resolve_symbols(spec)
        resolved[pool_label(spec)] = codes
        print(f"[{pool_label(spec)}] {len(codes)} 只 ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"[{pool_label(spec)}] 解析失败: {type(e).__name__}: {str(e)[:90]}")
    print()

# 一次拉取全市场名称(新浪快照),再按池展示
all_codes = sorted({c for codes in resolved.values() for c in codes})
names = symbol_names(all_codes)
for label, codes in resolved.items():
    print(f"{label}:")
    print("  " + "  ".join(f"{c}:{names.get(c, '?')}" for c in codes))
    print()

[沪深300成分股（前 10）] 10 只 (0.1s)

[data] Eastmoney spot unavailable — falling back to Sina daily (per-stock market cap over CSI300, ~30s–2min) …
[总市值最大（前 10）] 10 只 (23.4s)

[data] Eastmoney spot unavailable — falling back to Sina daily (per-stock market cap over CSI300, ~30s–2min) …
[总市值最小（前 10）] 10 只 (23.9s)

[行业(证监会) · 酒] 解析失败: Exception: 抱歉，您访问接口(stock_basic)频率超限(1次/小时)，具体频次详情：https://tushare.pro/document/1?doc_id=108。

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · PE(静) 最低（前 10）] 10 只 (0.0s)

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · PE(TTM) 最低（前 10）] 10 只 (0.0s)

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · PB 最高（前 10）] 10 只 (0.0s)

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · 股息率 最高（前 10）] 10 只 (0.0s)

[自选列表（5 只）] 5 只 (0.0s)

沪深300成分股（前 10）:
  600118:中国卫星  002532:天山铝业  000988:华工科技  000657:中钨高新  301308:江波龙  688072:拓荆科技  002202:金风科技  

[data] Eastmoney spot unavailable — falling back to Sina daily (per-stock market cap over CSI300, ~30s–2min) …
[总市值最小（前 10）] 10 只 (20.2s)

[行业(证监会) · 酒] 解析失败: Exception: 抱歉，您访问接口(stock_basic)频率超限(5次/天)，具体频次详情：https://tushare.pro/document/1?doc_id=108。

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · PE(静) 最低（前 10）] 10 只 (0.0s)

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · PE(TTM) 最低（前 10）] 10 只 (0.0s)

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · PB 最高（前 10）] 10 只 (0.0s)

[data] Using cached valuation table at 'data/valuation/sw_valuation.parquet'
[估值 · 股息率 最高（前 10）] 10 只 (0.0s)

[自选列表（5 只）] 5 只 (0.0s)

沪深300成分股（前 10）:
  600118:中国卫星  002532:天山铝业  000988:华工科技  000657:中钨高新  301308:江波龙  688072:拓荆科技  002202:金风科技  301165:锐捷网络  002558:巨人网络  688183:生益电子

总市值最大（前 10）:
  601288:农业银行  601398:工商银行  601857:中国石油  600519:贵州茅台  300750:宁德时代  601988:中国银行  601138:工业富联  300308:中际旭创  600036:招商银行 

## 3. 估值池(valuation:)详解

数据源:tushare `daily_basic`(正式接口,一次调用返回全市场约 5500 只);
该接口不可用时回退乐咕申万三级成分页爬虫(口径近似,见 §3.4)。

### 3.1 各指标的计算方法

| 指标 | 项目内列名 | 计算公式 | 默认方向 |
|---|---|---|---|
| 静态市盈率 | `pe` | 总市值 ÷ 上年年度归母净利润(= 收盘价 ÷ 上年每股收益) | low |
| 市盈率(TTM) | `pe_ttm` | 总市值 ÷ 最近 4 个季度滚动归母净利润 | low |
| 市净率 | `pb` | 总市值 ÷ 最新归母净资产 | low |
| 股息率 | `dy` | 近 12 个月每股现金分红合计 ÷ 收盘价(百分数) | **high** |
| 总市值 | `cap` | 收盘价 × 总股本(亿元) | — |
| 流通市值 | `circ_cap` | 收盘价 × 流通股本(亿元) | — |

要点:

* **静态 PE 的分母是"去年整年"的利润**,即使现在已是下半年——盈利变化快的公司
  静态 PE 会失真;**TTM(Trailing Twelve Months)PE** 把分母换成最近连续 4 个
  季度的利润,反映最新盈利水平,成长股常用 `pe_ttm` 替代 `pe`;
* **PB** 适合金融/周期/重资产行业(盈利大幅波动时 PE 失真),净资产为负时 PB
  无意义(被过滤);
* **股息率**只算现金分红(不含送转股),tushare 的 `dv_ttm` 口径 = 近 12 个月
  每股分红 ÷ 当前价格;
* **单位换算**:tushare 的 `total_mv`/`circ_mv` 原单位是**万元**,入库时除以
  10⁴ 转为**亿元**(`cap`/`circ_cap`)。

### 3.2 为什么只保留正值

* 亏损公司(净利润为负)PE 为负,"PE 最低"会把最差的公司排到最前——所以
  `pe`/`pe_ttm` 只取 > 0(即"有正盈利");
* 净资产为负 → PB < 0,同样排除;无分红/分红为 0 → 股息率排除;
* 停牌、缺失值(`—`、`%` 占位、NaN)一律过滤。

`valuation:pe_low:10` 的准确语义 = **"有正盈利的 10 只最低静态 PE"**。

### 3.4 口径差异(乐咕兜底路径)

* 乐咕申万成分页的 `市盈率` = 总市值 ÷ 上年净利润,与 tushare 归母口径近似但
  可能略有舍入差异;`市值(亿元)`、`股息率` 口径相同;
* 两条路径写入同一份缓存(schema 一致),tushare 优先;乐咕表格没有流通市值,
  `circ_cap` 列仅在 tushare 路径存在(`market_cap` 的 float_cap 会因此自动
  改走其他来源)。

### 缓存

结果落盘 `data/valuation/sw_valuation.parquet`(含 crawl_date 列),同日再次调用
秒回(不再消耗 API 额度);tushare 的 stock_basic/daily_basic 在该积分档限频
1 次/小时,日级缓存是硬约束。乐咕兜底路径:行业代码列表单独缓存
(`sw_industries.parquet`),爬取失败时若有陈旧缓存则自动沿用。



In [3]:
# ── §3b 用真实数据演示指标关系(从缓存反推各指标口径) ──
cache = "data/valuation/sw_valuation.parquet"
if os.path.isfile(cache):
    t = pl.read_parquet(cache)
    row = t.filter(pl.col("code") == "600519")
    if row.height:
        r = row.row(0, named=True)
        cap = r["cap"]
        print(f"贵州茅台(600519):总市值 {cap:,.0f} 亿元")
        pe, pe_ttm, pb, dy = r["pe"], r["pe_ttm"], r["pb"], r["dy"]
        if pe and pe > 0:
            print(f"  静态 PE = 总市值 / 上年净利润 = {cap:,.0f} / {cap/pe:,.0f} ≈ {pe:.1f}")
        if pe_ttm and pe_ttm > 0:
            print(f"  PE(TTM)= 总市值 / 近4季净利润 = {cap:,.0f} / {cap/pe_ttm:,.0f} ≈ {pe_ttm:.1f}")
        if pb and pb > 0:
            print(f"  PB     = 总市值 / 净资产     = {cap:,.0f} / {cap/pb:,.0f} ≈ {pb:.2f}")
        if dy and dy > 0:
            print(f"  股息率 ≈ 近12月分红 / 总市值  = {cap*dy/100:,.0f} / {cap:,.0f} ≈ {dy:.2f}%")
    else:
        print("缓存中无 600519")
else:
    print("暂无估值缓存 — 首次运行 valuation:/market_cap: 规格后自动落盘")


贵州茅台(600519):总市值 15,941 亿元
  静态 PE = 总市值 / 上年净利润 = 15,941 / 823 ≈ 19.4
  PE(TTM)= 总市值 / 近4季净利润 = 15,941 / 814 ≈ 19.6
  PB     = 总市值 / 净资产     = 15,941 / 2,513 ≈ 6.34
  股息率 ≈ 近12月分红 / 总市值  = 650 / 15,941 ≈ 4.08%


In [4]:
# ── §4 估值缓存检查 ──
cache = "data/valuation/sw_valuation.parquet"
if os.path.isfile(cache):
    t = pl.read_parquet(cache)
    print(f"估值缓存: {t.shape[0]} 只, crawl_date = {t['crawl_date'].unique().to_list()}")
    # 估值缓存里的 name 可能为 null(写缓存时 stock_basic 恰被限频);
    # 若有股票基础信息缓存,显示时补上名称。
    sb = "data/universe/stock_basic.parquet"
    show = t
    if os.path.isfile(sb) and t["name"].null_count() == t.height:
        show = (t.drop("name")
                 .join(pl.read_parquet(sb).select(["code", "name"]),
                       on="code", how="left"))
    print("样本(PE 最低 8 只):")
    print(show.filter(show["pe"] > 0).sort("pe").head(8)
           .select(["code", "name", "pe", "pe_ttm", "pb", "dy", "cap"]))
    print()
    for col in ["pe", "pe_ttm", "pb", "dy"]:
        valid = t[col].is_finite() & (t[col] > 0)
        s = t[col].filter(valid)
        print(f"{col:>6}: 有效 {valid.sum():>5} 只, 中位数 {s.median():>8.2f}")
else:
    print("暂无估值缓存 — 首次运行 valuation:/market_cap: 规格后自动落盘(tushare 或乐咕)")
ind_cache = "data/valuation/sw_industries.parquet"
if os.path.isfile(ind_cache):
    print(f"行业代码列表缓存: {pl.read_parquet(ind_cache).shape[0]} 个三级行业")


估值缓存: 5550 只, crawl_date = [20260912]
样本(PE 最低 8 只):
shape: (8, 7)
┌────────┬──────────┬────────┬────────┬────────┬────────┬─────────────┐
│ code   ┆ name     ┆ pe     ┆ pe_ttm ┆ pb     ┆ dy     ┆ cap         │
│ ---    ┆ ---      ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---         │
│ str    ┆ str      ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64         │
╞════════╪══════════╪════════╪════════╪════════╪════════╪═════════════╡
│ 000656 ┆ 金科股份 ┆ 0.4369 ┆ 0.3475 ┆ 3.069  ┆ null   ┆ 128.123652  │
│ 600841 ┆ 动力新科 ┆ 2.7834 ┆ 2.458  ┆ 1.2381 ┆ null   ┆ 79.938536   │
│ 000498 ┆ 山东路桥 ┆ 3.4155 ┆ 3.9422 ┆ 0.4219 ┆ 4.5039 ┆ 76.069658   │
│ 600015 ┆ 华夏银行 ┆ 3.7154 ┆ 4.0271 ┆ 0.3155 ┆ 6.6142 ┆ 1010.59796  │
│ 601997 ┆ 贵阳银行 ┆ 3.9145 ┆ 4.0014 ┆ 0.3112 ┆ 5.3381 ┆ 205.478333  │
│ 601186 ┆ 中国铁建 ┆ 4.5111 ┆ 5.0661 ┆ 0.2984 ┆ 4.918  ┆ 828.352032  │
│ 601818 ┆ 光大银行 ┆ 4.5958 ┆ 5.4212 ┆ 0.3516 ┆ 5.7947 ┆ 1784.383643 │
│ 601668 ┆ 中国建筑 ┆ 4.6007 ┆ 5.6756 ┆ 0.3632 ┆ 6.2483 ┆ 1797.436982 │
└────────┴──────────┴────────

## 5. 数据源与降级路径

| 源 | 服务对象 | 已知问题 | 降级行为 |
|---|---|---|---|
| tushare | `industry:`(证监会行业)、`valuation:`、`market_cap:` 首选 | 该档位限频严(1次/分钟→1次/小时阶梯) | 日级缓存;估值回退乐咕爬虫;无 token 时跳过。**token 配置**:`export TUSHARE_TOKEN=...` 或写入 `data/tushare_token.txt`(已被 .gitignore 忽略) |
| 新浪 | 日线(`stock_zh_a_daily`)、全市场快照(名称)、market_cap 兜底 | 快照逐页拉取较慢(约 20s);兜底逐股慢 | 名称失败显示 `?`;兜底结果每日缓存于 `data/universe/` |
| 乐咕 | `valuation:` 兜底(申万三级成分页爬虫) | 偶发全站 504 / 页面改版(曾导致 akshare 封装失效) | 每日缓存 + 页面级重试 + 陈旧缓存兜底 |
| 中证 | `csi300:` | 稳定 | — |

诊断现状:东财 push2 被限流;tushare 可用但限频严;新浪、中证正常;乐咕曾全站
504。行业池已切 tushare 证监会行业(东财/乐咕行业成分不再使用)。

In [5]:
# ── §6 可选:为最后一个池子拉取日线并缓存 ──
if FETCH_BARS:
    spec = POOL_SPECS[-1]
    codes = resolve_symbols(spec)
    end = datetime.date.today().strftime("%Y%m%d")
    start = (datetime.date.today() - datetime.timedelta(days=365)).strftime("%Y%m%d")
    df = fetch_daily_to_parquet(codes, start, end, root="data/pool_demo")
    print(f"{pool_label(spec)} 日线: {df.shape[0]} 行, "
          f"{df['symbol'].n_unique()} 只 × {df['date'].n_unique()} 天")
    print(df.tail(4))
else:
    print("FETCH_BARS=False — 设置 True 重新运行本 cell")

FETCH_BARS=False — 设置 True 重新运行本 cell


## 7. 与回测 / 实盘的关系

股票池只决定 **universe**(截面里有哪些股票);信号、组合、风控、执行层都与池子无关:

* 回测:demo.ipynb 中 `STOCK_POOL` 决定 `data/real_daily` 缓存的池子,`DATA_SOURCE`
  指向缓存后跑 §6+ 全流程。
* 实盘模拟:`python -m alftrade.paper --pool "valuation:pe_low:30"` — `--pool`
  接受本 notebook 里全部规格;`--show-cron` 查看定时任务。

常见组合:先用某个池子 + 30 天回测看行为,再换 `--pool` 跑 paper 模拟。